In [21]:
# ─── Cell 2: Imports & Configuration ─────────────────────────────────────────
import os
import gradio as gr
from openai import OpenAI

# Set your key via env var before running:  export OPENROUTER_API_KEY=sk-or-v1-...
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise EnvironmentError("OPENROUTER_API_KEY is not set. Run: export OPENROUTER_API_KEY=sk-or-v1-...")

AVAILABLE_MODELS = [
    "anthropic/claude-sonnet-4-5",
    "anthropic/claude-3-5-haiku",
    "anthropic/claude-3-opus",
    "openai/gpt-4o",
    "openai/gpt-4o-mini",
    "google/gemini-2.0-flash-001",
    "meta-llama/llama-3.3-70b-instruct",
    "mistralai/mistral-large",
    "deepseek/deepseek-chat-v3-0324",
    "qwen/qwen-2.5-coder-32b-instruct",
]
DEFAULT_MODEL = AVAILABLE_MODELS[0]

LANGUAGES = [
    "Auto-Detect",
    "Python", "JavaScript", "TypeScript", "Java", "C", "C++",
    "C#", "Go", "Rust", "Ruby", "PHP", "Swift", "Kotlin",
    "R", "Scala", "Haskell", "Lua", "Perl", "Shell/Bash",
]

STYLES = {
    "Python":     ["Google Style", "NumPy Style", "reStructuredText (Sphinx)", "Epytext"],
    "JavaScript": ["JSDoc"],
    "TypeScript": ["TSDoc", "JSDoc"],
    "Java":       ["Javadoc"],
    "C":          ["Doxygen"],
    "C++":        ["Doxygen"],
    "C#":         ["XML Doc Comments"],
    "Go":         ["GoDoc"],
    "Rust":       ["RustDoc"],
    "Ruby":       ["YARD", "RDoc"],
    "PHP":        ["PHPDoc"],
    "Swift":      ["Swift Markup"],
    "Kotlin":     ["KDoc"],
    "R":          ["Roxygen2"],
    "Scala":      ["Scaladoc"],
    "default":    ["Standard Comments"],
}

print("Imports OK — ready.")

Imports OK — ready.


In [22]:
# ─── Cell 3: Core Logic ───────────────────────────────────────────────────────

def update_style_dropdown(language: str):
    if language == "Auto-Detect":
        choices = ["Standard Comments", "Google Style", "JSDoc", "Doxygen", "Javadoc"]
    else:
        choices = STYLES.get(language, STYLES["default"])
    return gr.Dropdown(choices=choices, value=choices[0])


def build_system_prompt(language, doc_style, add_inline, detail_level, include_examples, include_types):
    lang_str = "the programming language used in the code" if language == "Auto-Detect" else language
    lines = [
        f"You are an expert {lang_str} developer and technical writer.",
        f"Your task: add {doc_style} docstrings/documentation comments to the provided {lang_str} code.",
        f"Documentation detail level: {detail_level}.",
    ]
    if add_inline:
        lines.append("Also add concise inline comments for complex or non-obvious lines.")
    if include_examples:
        lines.append("Include a usage example in each function/method docstring where helpful.")
    if include_types:
        lines.append("Always document parameter types and return types.")
    lines += [
        "IMPORTANT: Return ONLY the fully documented source code.",
        "Do NOT wrap output in markdown code fences or add any prose outside the code.",
        "Preserve the original logic, indentation, and formatting exactly.",
    ]
    return "\n".join(lines)


def generate_docstrings(model, code, language, doc_style,
                        add_inline, detail_level, include_examples, include_types):
    if not code.strip():
        return "Please paste some code first."

    or_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    system = build_system_prompt(language, doc_style, add_inline, detail_level,
                                  include_examples, include_types)
    try:
        response = or_client.chat.completions.create(
            model=model,
            max_tokens=4096,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": code},
            ],
            extra_headers={
                "HTTP-Referer": "https://github.com/docstring-generator",
                "X-Title": "Docstring Generator",
            },
        )
        result = response.choices[0].message.content or ""
        # Strip accidental markdown fences
        if result.startswith("```"):
            lines = result.splitlines()
            result = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
        return result
    except Exception as exc:
        err = str(exc)
        if "401" in err or "authentication" in err.lower():
            return "Authentication error — check your OpenRouter API key."
        if "429" in err or "rate" in err.lower():
            return "Rate limit hit — please wait a moment and try again."
        if "402" in err or "credit" in err.lower():
            return "Insufficient credits on your OpenRouter account."
        return f"Unexpected error: {exc}"


def copy_to_input(text):
    return text


print("Functions defined.")

Functions defined.


In [23]:
# ─── Cell 4: Sample Snippets ──────────────────────────────────────────────────

SAMPLES = {
    "Python — fibonacci": (
        "Python",
        "def fibonacci(n, memo={}):\n"
        "    if n in memo:\n"
        "        return memo[n]\n"
        "    if n <= 1:\n"
        "        return n\n"
        "    memo[n] = fibonacci(n - 1, memo) + fibonacci(n - 2, memo)\n"
        "    return memo[n]\n\n"
        "def fib_sequence(length):\n"
        "    return [fibonacci(i) for i in range(length)]\n"
    ),
    "JavaScript — fetch helper": (
        "JavaScript",
        "async function fetchData(url, options = {}) {\n"
        "  const defaults = { method: 'GET', headers: { 'Content-Type': 'application/json' } };\n"
        "  const config = { ...defaults, ...options };\n"
        "  const response = await fetch(url, config);\n"
        "  if (!response.ok) throw new Error(`HTTP ${response.status}`);\n"
        "  return response.json();\n"
        "}\n"
    ),
    "Java — binary search": (
        "Java",
        "public class BinarySearch {\n"
        "    public static int search(int[] arr, int target) {\n"
        "        int left = 0, right = arr.length - 1;\n"
        "        while (left <= right) {\n"
        "            int mid = left + (right - left) / 2;\n"
        "            if (arr[mid] == target) return mid;\n"
        "            if (arr[mid] < target) left = mid + 1;\n"
        "            else right = mid - 1;\n"
        "        }\n"
        "        return -1;\n"
        "    }\n"
        "}\n"
    ),
    "Go — HTTP handler": (
        "Go",
        "func healthHandler(w http.ResponseWriter, r *http.Request) {\n"
        "    if r.Method != http.MethodGet {\n"
        "        http.Error(w, \"Method not allowed\", http.StatusMethodNotAllowed)\n"
        "        return\n"
        "    }\n"
        "    w.Header().Set(\"Content-Type\", \"application/json\")\n"
        "    json.NewEncoder(w).Encode(map[string]string{\"status\": \"ok\"})\n"
        "}\n"
    ),
    "Rust — config parser": (
        "Rust",
        "fn parse_config(path: &str) -> Result<Config, Box<dyn Error>> {\n"
        "    let contents = fs::read_to_string(path)?;\n"
        "    let config: Config = serde_json::from_str(&contents)?;\n"
        "    if config.port == 0 {\n"
        "        return Err(\"Port cannot be zero\".into());\n"
        "    }\n"
        "    Ok(config)\n"
        "}\n"
    ),
}

print("Samples loaded.")

Samples loaded.


In [26]:
# ─── Cell 5: Gradio UI ────────────────────────────────────────────────────────

CSS = """
.gradio-container { font-family: 'IBM Plex Mono', monospace; }
#title    { text-align: center; margin-bottom: 0.2rem; }
#subtitle { text-align: center; color: #6b7280; margin-bottom: 1.2rem; }
#generate-btn { background: #111827 !important; color: #f9fafb !important; }
#generate-btn:hover { background: #374151 !important; }
#code-input textarea, #code-output textarea {
    font-family: 'JetBrains Mono', 'Fira Code', 'Courier New', monospace !important;
    font-size: 0.83rem !important;
    line-height: 1.5 !important;
    white-space: pre !important;
    overflow-wrap: normal !important;
    overflow-x: auto !important;
}
"""

def load_sample(name):
    if name == "(none)":
        return "", "Auto-Detect"
    lang, code = SAMPLES[name]
    return code, lang


with gr.Blocks(css=CSS, title="Docstring Generator") as demo:

    gr.Markdown("## 🧠 AI Docstring & Comment Generator", elem_id="title")
    gr.Markdown("Powered by **OpenRouter** · Any model · Any language", elem_id="subtitle")

    with gr.Row():
        model_picker = gr.Dropdown(
            choices=AVAILABLE_MODELS,
            value=DEFAULT_MODEL,
            label="🤖 Model",
            allow_custom_value=True,
            scale=2,
        )

    with gr.Row():
        with gr.Column(scale=3):
            code_input = gr.Textbox(
                label="📋 Paste your code here",
                placeholder="Paste any code here...",
                lines=25,
                max_lines=80,
                elem_id="code-input",
                show_copy_button=True,
            )

            with gr.Accordion("⚙️ Documentation Options", open=True):
                with gr.Row():
                    language = gr.Dropdown(
                        choices=LANGUAGES, value="Auto-Detect", label="Language", scale=2
                    )
                    doc_style = gr.Dropdown(
                        choices=["Standard Comments", "Google Style", "JSDoc", "Doxygen", "Javadoc"],
                        value="Standard Comments", label="Doc Style", scale=2
                    )
                    detail_level = gr.Radio(
                        choices=["Minimal", "Standard", "Detailed"],
                        value="Standard", label="Detail Level"
                    )
                with gr.Row():
                    add_inline       = gr.Checkbox(label="Inline comments", value=True)
                    include_types    = gr.Checkbox(label="Include types",    value=True)
                    include_examples = gr.Checkbox(label="Usage examples",  value=False)

            with gr.Row():
                sample_picker = gr.Dropdown(
                    choices=["(none)"] + list(SAMPLES.keys()),
                    value="(none)", label="💡 Load a sample", scale=3
                )
                generate_btn = gr.Button(
                    "✨ Generate", variant="primary", scale=1, elem_id="generate-btn"
                )

        with gr.Column(scale=3):
            code_output = gr.Textbox(
                label="📝 Documented code",
                placeholder="Output will appear here...",
                lines=25,
                max_lines=80,
                elem_id="code-output",
                interactive=True,
                show_copy_button=True,
            )
            with gr.Row():
                copy_back_btn = gr.Button("↩️  Use output as input", scale=1)
                clear_btn = gr.ClearButton(
                    components=[code_input, code_output],
                    value="🗑️  Clear all", scale=1
                )

            gr.Markdown(
                "### 📖 Style quick-reference\n"
                "| Language | Style |\n"
                "|----------|-------|\n"
                "| Python   | Google / NumPy / Sphinx |\n"
                "| JS / TS  | JSDoc / TSDoc |\n"
                "| Java     | Javadoc |\n"
                "| C / C++  | Doxygen |\n"
                "| Go       | GoDoc |\n"
                "| Rust     | RustDoc |\n"
                "| Ruby     | YARD / RDoc |\n"
                "| PHP      | PHPDoc |\n"
            )

    gr.Markdown(
        "<center><small>"
        "Key via <code>OPENROUTER_API_KEY</code> env var · "
        "<a href='https://openrouter.ai/keys' target='_blank'>openrouter.ai/keys</a> · "
        "<a href='https://openrouter.ai/models' target='_blank'>openrouter.ai/models</a>"
        "</small></center>"
    )

    language.change(fn=update_style_dropdown, inputs=[language], outputs=[doc_style])

    sample_picker.change(
        fn=load_sample,
        inputs=[sample_picker],
        outputs=[code_input, language],
    )

    generate_btn.click(
        fn=generate_docstrings,
        inputs=[model_picker, code_input, language, doc_style,
                add_inline, detail_level, include_examples, include_types],
        outputs=[code_output],
    )

    copy_back_btn.click(
        fn=lambda text: text,
        inputs=[code_output],
        outputs=[code_input],
    )

print("UI built — launching…")
demo.launch()

UI built — launching…
* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.
